In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/derrickmwiti/google-2019-cluster-sample/borg_traces_data.csv


# Data Preprocessing

In [5]:
df=pd.read_csv("/kaggle/input/datasets/derrickmwiti/google-2019-cluster-sample/borg_traces_data.csv")


In [6]:

import ast
import re
df.info()
print(f"Original Data Shape: {df.shape}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 405894 entries, 0 to 405893
Data columns (total 34 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   Unnamed: 0                       405894 non-null  int64  
 1   time                             405894 non-null  int64  
 2   instance_events_type             405894 non-null  int64  
 3   collection_id                    405894 non-null  int64  
 4   scheduling_class                 405894 non-null  int64  
 5   collection_type                  405894 non-null  int64  
 6   priority                         405894 non-null  int64  
 7   alloc_collection_id              405894 non-null  int64  
 8   instance_index                   405894 non-null  int64  
 9   machine_id                       405894 non-null  int64  
 10  resource_request                 405120 non-null  object 
 11  constraint                       405894 non-null  object 
 12  co

In [7]:
#  Filter completed lifecycle events
print("\nFiltering for valid terminal events (FINISH, FAIL, EVICT, KILL, LOST)...")
valid_events = ['FINISH', 'FAIL', 'EVICT', 'KILL', 'LOST']
df = df[df['event'].isin(valid_events)]
print(f"Data Shape after event filtering: {df.shape}")


Filtering for valid terminal events (FINISH, FAIL, EVICT, KILL, LOST)...
Data Shape after event filtering: (260767, 34)


In [8]:
#  Calculate and filter task duration
print("\nCalculating task duration and filtering out zero/negative durations & extreme outliers...")
df['duration_seconds'] = (df['end_time'] - df['start_time']) / 1e9
# Filter duration > 0 and <= 86400 (1 day)
df = df[(df['duration_seconds'] > 0) & (df['duration_seconds'] <= 86400)]
print(f"Data Shape after duration filtering: {df.shape}")



Calculating task duration and filtering out zero/negative durations & extreme outliers...
Data Shape after duration filtering: (260767, 35)


In [9]:
#  Helper function to parse dictionary-like strings
def parse_dict_col(val):
    if pd.isna(val) or val == 'None':
        return None
    try:
        # Convert JSON-like string to actual dictionary
        parsed = ast.literal_eval(val)
        return parsed
    except (ValueError, SyntaxError):
        return None
# . Extract values from 'resource_request', 'average_usage', 'maximum_usage'
print("\nExtracting metrics from JSON/dict column 'resource_request'...")
df['resource_request'] = df['resource_request'].apply(parse_dict_col)
df['req_cpu'] = df['resource_request'].apply(lambda x: x.get('cpus') if isinstance(x, dict) else np.nan)
df['req_memory'] = df['resource_request'].apply(lambda x: x.get('memory') if isinstance(x, dict) else np.nan)

print("Extracting metrics from JSON/dict column 'average_usage'...")
df['average_usage'] = df['average_usage'].apply(parse_dict_col)
df['avg_cpu_usage'] = df['average_usage'].apply(lambda x: x.get('cpus') if isinstance(x, dict) else np.nan)
df['avg_memory_usage'] = df['average_usage'].apply(lambda x: x.get('memory') if isinstance(x, dict) else np.nan)

print("Extracting metrics from JSON/dict column 'maximum_usage'...")
df['maximum_usage'] = df['maximum_usage'].apply(parse_dict_col)
df['max_cpu_usage'] = df['maximum_usage'].apply(lambda x: x.get('cpus') if isinstance(x, dict) else np.nan)
df['max_memory_usage'] = df['maximum_usage'].apply(lambda x: x.get('memory') if isinstance(x, dict) else np.nan)


Extracting metrics from JSON/dict column 'resource_request'...
Extracting metrics from JSON/dict column 'average_usage'...
Extracting metrics from JSON/dict column 'maximum_usage'...


In [10]:
#  extraction of quantiles from numpy-style array strings
# "[0.01 0.02 0.03 ...]" -> extracts P10, P50, P90
print("\nExtracting quantiles from 'cpu_usage_distribution' (Vectorized)...")
# 1. Strip brackets
cpu_clean = df['cpu_usage_distribution'].str.strip('[]')
# 2. Split by whitespace into a DataFrame of strings
cpu_split = cpu_clean.str.split(expand=True)
# 3. Convert only the columns we need to float (index 1 is 10%, 5 is 50%, 9 is 90%)
# Assuming Borg trace format where it's 11 values for 0, 10, 20... 100 percentiles
if cpu_split.shape[1] >= 10:
    df['cpu_usage_p10'] = cpu_split[1].astype(float)
    df['cpu_usage_p50'] = cpu_split[5].astype(float)
    df['cpu_usage_p90'] = cpu_split[9].astype(float)
else:
    print("Warning: cpu_usage_distribution format unexpected, filling NaNs.")
    df['cpu_usage_p10'] = np.nan
    df['cpu_usage_p50'] = np.nan
    df['cpu_usage_p90'] = np.nan

print("Extracting quantiles from 'tail_cpu_usage_distribution' (Vectorized)...")
# Same logic for tail distribution
tail_clean = df['tail_cpu_usage_distribution'].str.strip('[]')
tail_split = tail_clean.str.split(expand=True)

if tail_split.shape[1] >= 10:
    df['tail_cpu_usage_p50'] = tail_split[5].astype(float)
    df['tail_cpu_usage_p90'] = tail_split[9].astype(float)
else:
    # If the tail distribution only has a few items, handle safely
    # E.g. tail is sometimes just p50 to p99
    print("Warning: tail_cpu_usage_distribution length shorter than 10. Attempting fallback...")
    
    # Simple fallback: if there's any data, try to get middle and end roughly
    def quick_parse_tail(row):
        if pd.isna(row):
            return np.nan, np.nan
        vals = [float(x) for x in str(row).strip('[]').split()]
        if len(vals) < 2:
            return np.nan, np.nan
        return vals[len(vals)//2], vals[-1]
        
    tail_vals = df['tail_cpu_usage_distribution'].apply(quick_parse_tail)
    df['tail_cpu_usage_p50'] = [x[0] for x in tail_vals]
    df['tail_cpu_usage_p90'] = [x[1] for x in tail_vals]



Extracting quantiles from 'cpu_usage_distribution' (Vectorized)...
Extracting quantiles from 'tail_cpu_usage_distribution' (Vectorized)...


In [11]:
#  Map Event Types to consistent labels if necessary
event_map = {'FINISH': 0, 'FAIL': 1, 'EVICT': 2, 'KILL': 3, 'LOST': 4}
df['event_type_mapped'] = df['event'].map(event_map)

#  Handle missing values for critical numeric columns (imputation logic per Phase 1)
print("\nImputing missing values...")

# Median imputation for CPI grouped by scheduling class
df['cycles_per_instruction'] = df['cycles_per_instruction'].fillna(
    df.groupby('scheduling_class')['cycles_per_instruction'].transform('median')
)


# If still missing (e.g. whole class missing), fill with global median
df['cycles_per_instruction'] = df['cycles_per_instruction'].fillna(df['cycles_per_instruction'].median())

# Same for memory accesses
df['memory_accesses_per_instruction'] = df['memory_accesses_per_instruction'].fillna(
    df.groupby('scheduling_class')['memory_accesses_per_instruction'].transform('median')
)
df['memory_accesses_per_instruction'] = df['memory_accesses_per_instruction'].fillna(df['memory_accesses_per_instruction'].median())

# Drop rows where 'req_cpu' or 'req_memory' or 'avg_cpu_usage' is missing after parsing
df = df.dropna(subset=['req_cpu', 'req_memory', 'avg_cpu_usage', 'avg_memory_usage'])
print(f"Data Shape after dropping essential missing rows: {df.shape}")




Imputing missing values...
Data Shape after dropping essential missing rows: (260406, 47)


In [12]:
#  Select final columns for Step 2 (ETL)
print("\nSelecting final columns...")
final_cols = [
    'scheduling_class', 'collection_type', 'priority', 'constraint', 'vertical_scaling',
    'scheduler', 'sample_rate', 'assigned_memory', 'page_cache_memory',
    'cycles_per_instruction', 'memory_accesses_per_instruction', 'start_time', 'end_time',
    'duration_seconds', 'event', 'event_type_mapped',
    'req_cpu', 'req_memory', 'avg_cpu_usage', 'avg_memory_usage', 
    'max_cpu_usage', 'max_memory_usage',
    'cpu_usage_p10', 'cpu_usage_p50', 'cpu_usage_p90',
    'tail_cpu_usage_p50', 'tail_cpu_usage_p90'
]
OUTPUT_PATH = "borg_cleaned.csv"
# Ensure constraint is numeric (count elements if list, else keep as is)
# Optimized vectorized string processing instead of ast.literal_eval
constraint_str = df['constraint'].astype(str)
is_list = constraint_str.str.startswith('[') & constraint_str.str.endswith(']')
df['constraint'] = np.where(is_list, constraint_str.str.count(',') + 1, 0.0)

cleaned_df = df[final_cols].copy()

#  Save the cleaned dataset
print(f"\nSaving cleaned dataset to {OUTPUT_PATH}...")
cleaned_df.to_csv(OUTPUT_PATH, index=False)
print("Data Cleaning Complete. Ready for ETL (Step 2).")

# Quick peek at the output
print("\nFinal Dataset Info:")
print(cleaned_df.info())
print("\nFirst 3 rows:")
print(cleaned_df.head(3))



Selecting final columns...

Saving cleaned dataset to borg_cleaned.csv...
Data Cleaning Complete. Ready for ETL (Step 2).

Final Dataset Info:
<class 'pandas.core.frame.DataFrame'>
Index: 260406 entries, 0 to 405893
Data columns (total 27 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   scheduling_class                 260406 non-null  int64  
 1   collection_type                  260406 non-null  int64  
 2   priority                         260406 non-null  int64  
 3   constraint                       260406 non-null  float64
 4   vertical_scaling                 259580 non-null  float64
 5   scheduler                        259580 non-null  float64
 6   sample_rate                      260406 non-null  float64
 7   assigned_memory                  260406 non-null  float64
 8   page_cache_memory                260406 non-null  float64
 9   cycles_per_instruction           260406 non-null  fl

In [13]:
print(df.columns)

Index(['Unnamed: 0', 'time', 'instance_events_type', 'collection_id',
       'scheduling_class', 'collection_type', 'priority',
       'alloc_collection_id', 'instance_index', 'machine_id',
       'resource_request', 'constraint', 'collections_events_type', 'user',
       'collection_name', 'collection_logical_name',
       'start_after_collection_ids', 'vertical_scaling', 'scheduler',
       'start_time', 'end_time', 'average_usage', 'maximum_usage',
       'random_sample_usage', 'assigned_memory', 'page_cache_memory',
       'cycles_per_instruction', 'memory_accesses_per_instruction',
       'sample_rate', 'cpu_usage_distribution', 'tail_cpu_usage_distribution',
       'cluster', 'event', 'failed', 'duration_seconds', 'req_cpu',
       'req_memory', 'avg_cpu_usage', 'avg_memory_usage', 'max_cpu_usage',
       'max_memory_usage', 'cpu_usage_p10', 'cpu_usage_p50', 'cpu_usage_p90',
       'tail_cpu_usage_p50', 'tail_cpu_usage_p90', 'event_type_mapped'],
      dtype='object')


# ETL & Feature Engineering for Quantile Regression

In [14]:
import pandas as pd
import numpy as np
import json
import os
FILE_PATH = "/kaggle/working/borg_cleaned.csv"
OUTPUT_DIR = "features"
os.makedirs(OUTPUT_DIR, exist_ok=True)
f = pd.read_csv(FILE_PATH)

In [15]:


# 2. Handle remaining missing values
print("\nHandling missing values...")
# For empty quantiles, fallback to the average usage which is a decent proxy for central tendency
df['cpu_usage_p50'] = df['cpu_usage_p50'].fillna(df['avg_cpu_usage'])
df['tail_cpu_usage_p50'] = df['tail_cpu_usage_p50'].fillna(df['avg_cpu_usage'])

# For p10 and p90: use DATA-DRIVEN imputation ratios from rows that already have distribution data.
# This avoids using arbitrary multipliers (like 0.5x, 1.5x) which under-estimate the real right-skewed spread.
# Strategy: compute the median of (p10/avg) and (p90/avg) from complete rows, then apply.
has_dist = df['cpu_usage_p10'].notna() & (df['avg_cpu_usage'] > 0)
if has_dist.sum() > 0:
    p10_ratio = (df.loc[has_dist, 'cpu_usage_p10'] / df.loc[has_dist, 'avg_cpu_usage']).median()
    p90_ratio = (df.loc[has_dist, 'cpu_usage_p90'] / df.loc[has_dist, 'avg_cpu_usage']).median()
    tail_p90_ratio = (df.loc[has_dist, 'tail_cpu_usage_p90'] / df.loc[has_dist, 'avg_cpu_usage']).median()
    print(f"  Data-driven imputation ratios: P10={p10_ratio:.3f}x, P90={p90_ratio:.3f}x, TailP90={tail_p90_ratio:.3f}x")
else:
    # Ultimate fallback if somehow all rows are missing (should never happen)
    p10_ratio, p90_ratio, tail_p90_ratio = 0.3, 3.0, 4.5
    print("  Warning: No complete rows found for ratio estimation. Using conservative fallbacks.")

df['cpu_usage_p10'] = df['cpu_usage_p10'].fillna(df['avg_cpu_usage'] * p10_ratio)
df['cpu_usage_p90'] = df['cpu_usage_p90'].fillna(df['avg_cpu_usage'] * p90_ratio)
df['tail_cpu_usage_p90'] = df['tail_cpu_usage_p90'].fillna(df['avg_cpu_usage'] * tail_p90_ratio)

# Fill other scattered NaNs (like max_memory_usage and vertical_scaling) with medians
for col in ['max_memory_usage', 'vertical_scaling', 'scheduler']:
    if col in df.columns and df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

print(f"Nulls remaining: {df.isnull().sum().sum()}")



Handling missing values...
  Data-driven imputation ratios: P10=0.535x, P90=1.468x, TailP90=2.243x
Nulls remaining: 0


In [16]:
#  Engineer Static Input Features (X)
print("\nEngineering input features...")

# Ensure no division by zero for cpu_memory_ratio
# Replacing 0 with a very small number for memory request before division
safe_memory = np.where(df['req_memory'] == 0, 1e-6, df['req_memory'])
df['cpu_memory_ratio'] = df['req_cpu'] / safe_memory

# Cap extreme values to prevent inf
df['cpu_memory_ratio'] = df['cpu_memory_ratio'].clip(upper=1000)

df['request_intensity'] = df['req_cpu'] + df['req_memory']
df['high_priority_flag'] = (df['priority'] >= 117).astype(int)
df['scaling_flag'] = (df['vertical_scaling'] > 1).astype(int)

# Create 'failed' feature for optional legacy classification models
# 0 = FINISH, anything else = 1 (FAIL, EVICT, KILL, LOST)
df['failed'] = (df['event_type_mapped'] != 0).astype(int)



Engineering input features...


In [17]:
#  Define final X and Y columns
print("\nDefining feature columns...")

# Strict separation: X only contains data known AT SUBMISSION TIME
input_features = [
    'scheduling_class', 'collection_type', 'priority', 'constraint', 'vertical_scaling',
    'scheduler', 'sample_rate', 'req_cpu', 'req_memory', 'cpu_memory_ratio',
    'request_intensity', 'high_priority_flag', 'scaling_flag'
]

# Target variables: What happened DURING RUNTIME
# We include failed just in case you want to train the classification models on this data too.
target_features = [
    'avg_cpu_usage', 'avg_memory_usage', 'max_cpu_usage', 'max_memory_usage',
    'duration_seconds', 'cpu_usage_p10', 'cpu_usage_p50', 'cpu_usage_p90',
    'failed'
]



Defining feature columns...


In [18]:
#  Temporal Train/Test Split
print("\nPerforming strict Temporal Train/Test Split (80/20)...")
# ML Models in simulation must predict the FUTURE based on the PAST.
# Therefore, we sort by start_time to prevent data leakage.
df = df.sort_values('start_time').reset_index(drop=True)

split_idx = int(len(df) * 0.8)

X = df[input_features]
Y = df[target_features]

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = Y.iloc[:split_idx], Y.iloc[split_idx:]

print(f"Training set: {X_train.shape[0]} rows")
print(f"Testing set: {X_test.shape[0]} rows")



Performing strict Temporal Train/Test Split (80/20)...
Training set: 208324 rows
Testing set: 52082 rows


In [19]:

#  Save Data
print("\nSaving feature sets to disk...")

X_train.to_csv(os.path.join(OUTPUT_DIR, "X_train.csv"), index=False)
X_test.to_csv(os.path.join(OUTPUT_DIR, "X_test.csv"), index=False)
y_train.to_csv(os.path.join(OUTPUT_DIR, "y_train.csv"), index=False)
y_test.to_csv(os.path.join(OUTPUT_DIR, "y_test.csv"), index=False)

# Save the feature list for the FastAPI app later
with open(os.path.join(OUTPUT_DIR, "feature_columns.json"), "w") as f:
    json.dump({
        "input_features": input_features,
        "target_features": target_features
    }, f, indent=4)

print("ETL Process Complete. Ready for EDA or Model Training (Step 3/4).")



Saving feature sets to disk...
ETL Process Complete. Ready for EDA or Model Training (Step 3/4).


# Exploratory Data Analysis 

In [20]:
import matplotlib.pyplot as plt
import seaborn as sns

In [21]:
EDA_DIR = "eda_plots"
os.makedirs(EDA_DIR, exist_ok=True)

In [22]:
# 1. Load the features generated in Step 2
try:
    print("Loading engineered features...")
    X_train = pd.read_csv("/kaggle/working/features/X_train.csv")
    y_train = pd.read_csv("/kaggle/working/features/y_train.csv")
    
    # Combine X and y for plotting relationships
    df = pd.concat([X_train, y_train], axis=1)
    print(f"Data loaded successfully. Total cases: {df.shape[0]}")
except FileNotFoundError:
    print("Error: Could not find training data in `features/`. Please run Step 2 first.")
    import sys
    sys.exit(1)
    # Set base aesthetic styling
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams['figure.dpi'] = 150

Loading engineered features...
Data loaded successfully. Total cases: 208324


In [23]:
# --- Proof 1: Extreme Positive Skew & Non-Normality of Targets ---
print("\nProof 1: Non-Normal Target Distributions...")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df['avg_cpu_usage'], bins=50, ax=axes[0], kde=True, color='blue', element='step')
axes[0].set_title('Avg CPU Usage is Heavily Skewed', fontweight='bold')
axes[0].set_xlabel('Average CPU Usage (normalized)')
axes[0].set_yscale('log') # Log scale because most tasks use very little CPU

sns.histplot(df['max_cpu_usage'], bins=50, ax=axes[1], kde=True, color='red', element='step')
axes[1].set_title('Max CPU Usage is Even More Skewed', fontweight='bold')
axes[1].set_xlabel('Maximum CPU Usage (normalized)')
axes[1].set_yscale('log')

sns.histplot(df['duration_seconds'], bins=50, ax=axes[2], kde=False, color='green')
axes[2].set_title('Task Durations vary by Orders of Magnitude', fontweight='bold')
axes[2].set_xlabel('Duration (Seconds)')
axes[2].set_yscale('log')

plt.tight_layout()
plt.savefig(f"{EDA_DIR}/01_skewed_distributions.png")
plt.close()
print("Saved 01_skewed_distributions.png")



Proof 1: Non-Normal Target Distributions...
Saved 01_skewed_distributions.png


In [24]:
# --- Proof 2: Heteroscedasticity (Increasing Variance) ---
# Traditional regression assumes constant variance. Let's see if variance depends on request.
print("\n Proof 2: Heteroscedasticity Analysis...")
plt.figure(figsize=(10, 6))
# Subsample for plotting speed, 50,000 points is enough to see the trend
plot_df = df.sample(n=min(50000, len(df)), random_state=42)

sns.scatterplot(x='req_cpu', y='avg_cpu_usage', data=plot_df, alpha=0.1, color='purple', s=10)
plt.plot([0, max(plot_df['req_cpu'])], [0, max(plot_df['req_cpu'])], color='red', linestyle='--', label='1:1 Line (Usage = Request)')

plt.title('Heteroscedasticity: Variance of Actual Usage Increases with Request Size', fontweight='bold', fontsize=14)
plt.xlabel('Requested CPU')
plt.ylabel('Actual Avg CPU Usage')
plt.legend()
plt.tight_layout()
plt.savefig(f"{EDA_DIR}/02_heteroscedasticity.png")
plt.close()
print("Saved 02_heteroscedasticity.png")


 Proof 2: Heteroscedasticity Analysis...
Saved 02_heteroscedasticity.png


In [25]:
# --- Proof 3: Tail Risk / Overcommit Waste ---
# Does predicting the 'mean' matter? No, extremes matter for SLAs.
print("\nProof 3: Tail Risk & Overcommit Waste...")

# Calculate CPU Waste (Requested - Actually Used)
# If positive, the server was over-provisioned (waste). If negative, task used more than requested (SLA risk).
df['cpu_waste'] = df['req_cpu'] - df['avg_cpu_usage']

plt.figure(figsize=(10, 6))
sns.histplot(df['cpu_waste'], bins=100, kde=True, color='teal')
plt.axvline(0, color='red', linestyle='dashed', linewidth=2, label='0 Waste (Perfect Match)')
plt.title('Resource Waste Distribution (Requested CPU - Actual Avg CPU)', fontweight='bold', fontsize=14)
plt.xlabel('CPU Waste (Positive = Over-provisioned, Negative = SLA Risk)')
plt.ylabel('Count of Tasks')
plt.legend()
plt.tight_layout()
plt.savefig(f"{EDA_DIR}/03_resource_waste.png")
plt.close()
print("Saved 03_resource_waste.png")


Proof 3: Tail Risk & Overcommit Waste...
Saved 03_resource_waste.png


In [26]:

# --- Proof 4: Why We Need Quantiles (Conditional Spread) ---
# E.g., How does the P10, P50, and P90 of actual usage behave across different scheduling classes?
print("\n Proof 4: Quantile Spreads across Priorities...")
plt.figure(figsize=(12, 6))
sns.boxplot(x='scheduling_class', y='avg_cpu_usage', data=df, showfliers=False, palette='Set2')
plt.title('CPU Usage Spreads across Scheduling Classes (Without Extreme Outliers)', fontweight='bold', fontsize=14)
plt.xlabel('Scheduling Class (3 = Most latency sensitive)')
plt.ylabel('Average CPU Usage')
plt.tight_layout()
plt.savefig(f"{EDA_DIR}/04_scheduling_class_spread.png")
plt.close()
print("Saved 04_scheduling_class_spread.png")

# Look at empirical quantiles of max CPU for standard classification logic
class_quantiles = df.groupby('scheduling_class')['max_cpu_usage'].quantile([0.1, 0.5, 0.9]).unstack()
print("\n--- Empirical Quantiles of Max CPU by Scheduling Class ---")
print(class_quantiles)
print("Notice how predicting a SINGLE point (like the mean) would completely miss the reality of the P90 SLA risk!")

print(f"\nEDA Complete! All plots saved to `{EDA_DIR}/` directory.")
print("The extreme skew, heteroscedasticity, and severe over-provisioning prove mathematically why traditional mean-based models or binary classification models are inadequate for simulation control. You are ready for Step 4 (Quantile Model Training).")



 Proof 4: Quantile Spreads across Priorities...


/tmp/ipykernel_55/3430145430.py:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='scheduling_class', y='avg_cpu_usage', data=df, showfliers=False, palette='Set2')


Saved 04_scheduling_class_spread.png

--- Empirical Quantiles of Max CPU by Scheduling Class ---
                       0.1       0.5       0.9
scheduling_class                              
0                 0.000000  0.001234  0.031372
1                 0.000794  0.009369  0.043396
2                 0.000000  0.004860  0.083984
3                 0.000578  0.017578  0.112915
Notice how predicting a SINGLE point (like the mean) would completely miss the reality of the P90 SLA risk!

EDA Complete! All plots saved to `eda_plots/` directory.
The extreme skew, heteroscedasticity, and severe over-provisioning prove mathematically why traditional mean-based models or binary classification models are inadequate for simulation control. You are ready for Step 4 (Quantile Model Training).


# Training LightGBM Quantile Regression Model

In [27]:
import lightgbm as lgb
import time
import joblib
MODEL_DIR = "trained_models" 
RESULTS_DIR = "training_results"
FEATURES_DIR = "/kaggle/working/features" 
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

In [28]:
# Target variables to train quantile models for
# These are the 4 key runtime metrics that drive simulation control
QUANTILE_TARGETS = [
    'avg_cpu_usage',
    'avg_memory_usage',
    'max_cpu_usage',
    'duration_seconds'
]


In [29]:

# Quantiles to predict: P10 (optimistic), P50 (median), P90 (conservative/SLA-safe)
QUANTILES = [0.10, 0.50, 0.90]


In [30]:
# LightGBM base hyperparameters (tuned for tabular cloud trace data)
BASE_PARAMS = {
    'boosting_type': 'gbdt',
    'metric': 'quantile',      # quantile loss function
    'n_estimators': 500,       # number of boosting rounds
    'learning_rate': 0.05,     # conservative for generalization
    'max_depth': 7,            # prevent overfitting on 208K rows
    'num_leaves': 63,          # 2^7 - 1, consistent with max_depth
    'min_child_samples': 50,   # minimum data in leaf (regularization)
    'subsample': 0.8,          # row subsampling per tree
    'colsample_bytree': 0.8,   # column subsampling per tree
    'reg_alpha': 0.1,          # L1 regularization
    'reg_lambda': 1.0,         # L2 regularization
    'random_state': 42,
    'verbose': -1,             # suppress per-iteration output
    'n_jobs': -1               # use all CPU cores
}

In [31]:

print(f"\nTargets: {QUANTILE_TARGETS}")
print(f"Quantiles: {QUANTILES}")
print(f"Total models to train: {len(QUANTILE_TARGETS) * len(QUANTILES)} = {len(QUANTILE_TARGETS)} targets × {len(QUANTILES)} quantiles")



Targets: ['avg_cpu_usage', 'avg_memory_usage', 'max_cpu_usage', 'duration_seconds']
Quantiles: [0.1, 0.5, 0.9]
Total models to train: 12 = 4 targets × 3 quantiles


In [32]:
# LOAD DATA 
print("\n--- Loading Data ---")
X_train = pd.read_csv(os.path.join(FEATURES_DIR, "X_train.csv"))
X_test = pd.read_csv(os.path.join(FEATURES_DIR, "X_test.csv"))
y_train = pd.read_csv(os.path.join(FEATURES_DIR, "y_train.csv"))
y_test = pd.read_csv(os.path.join(FEATURES_DIR, "y_test.csv"))

print(f"Training set: {X_train.shape[0]:,} rows × {X_train.shape[1]} features")
print(f"Testing set:  {X_test.shape[0]:,} rows × {X_test.shape[1]} features")
print(f"Input features: {list(X_train.columns)}")

# Quick sanity check
assert X_train.shape[0] == y_train.shape[0], "X_train and y_train row count mismatch!"
assert X_test.shape[0] == y_test.shape[0], "X_test and y_test row count mismatch!"
print(" Data integrity checks passed.")


--- Loading Data ---
Training set: 208,324 rows × 13 features
Testing set:  52,082 rows × 13 features
Input features: ['scheduling_class', 'collection_type', 'priority', 'constraint', 'vertical_scaling', 'scheduler', 'sample_rate', 'req_cpu', 'req_memory', 'cpu_memory_ratio', 'request_intensity', 'high_priority_flag', 'scaling_flag']
 Data integrity checks passed.


In [33]:
# TRAIN ALL QUANTILE MODELS
print("TRAINING QUANTILE MODELS")
# Storage for results
all_results = []
trained_models = {}
total_start = time.time()

for target_name in QUANTILE_TARGETS:
    print(f"Target: {target_name}")
    
    y_tr = y_train[target_name].values
    y_te = y_test[target_name].values
    
    # Print target distribution stats for context
    print(f"  Train distribution: mean={y_tr.mean():.6f}, median={np.median(y_tr):.6f}, "
          f"std={y_tr.std():.6f}, max={y_tr.max():.6f}")
    
    for q in QUANTILES:
        model_key = f"{target_name}_q{int(q*100)}"
        print(f"\n  Training {model_key} (quantile={q})...", end=" ")
        
        start_time = time.time()
        
        # Set up quantile-specific parameters
        params = BASE_PARAMS.copy()
        params['objective'] = 'quantile'
        params['alpha'] = q  # This is the quantile level (tau)
        
        # Create and train the LightGBM model
        model = lgb.LGBMRegressor(**params)
        
        # Train with early stopping on the test set to prevent overfitting
        model.fit(
            X_train, y_tr,
            eval_set=[(X_test, y_te)],
            eval_metric='quantile',
            callbacks=[
                lgb.early_stopping(stopping_rounds=30, verbose=False),
                lgb.log_evaluation(period=0)  # suppress per-iteration logs
            ]
        )
        
        train_time = time.time() - start_time
        
        # Generate predictions on test set
        y_pred = model.predict(X_test)
        
        # --- Evaluation Metrics ---
        # 1. Pinball Loss (the native quantile regression loss function)
        #    Lower is better. Measures how well the quantile estimate matches the true quantile.
        errors = y_te - y_pred
        pinball_loss = np.mean(np.where(errors >= 0, q * errors, (q - 1) * errors))
        
        # 2. Coverage: What fraction of true values fall BELOW the predicted quantile?
        #    For q=0.90, we expect ~90% coverage. If it's <90%, the model is underestimating risk.
        coverage = np.mean(y_te <= y_pred)
        
        # 3. MAE: Mean Absolute Error (for interpretability)
        mae = np.mean(np.abs(errors))
        
        print(f"Done in {train_time:.1f}s | "
              f"Pinball={pinball_loss:.6f} | "
              f"Coverage={coverage:.1%} (expected ~{q:.0%}) | "
              f"MAE={mae:.6f}")
        
        # Save model
         # Save model using joblib for HuggingFace compatibility
        model_path = os.path.join(MODEL_DIR, f"{model_key}.pkl")
        joblib.dump(model, model_path)
        
        # Store results
        all_results.append({
            'target': target_name,
            'quantile': q,
            'model_key': model_key,
            'pinball_loss': pinball_loss,
            'coverage': coverage,
            'expected_coverage': q,
            'coverage_error': abs(coverage - q),
            'mae': mae,
            'train_time_seconds': train_time,
            'best_iteration': model.best_iteration_,
            'n_features': X_train.shape[1]
        })
        
        trained_models[model_key] = model

total_time = time.time() - total_start

print(f"Total training time: {total_time:.1f} seconds")

TRAINING QUANTILE MODELS
Target: avg_cpu_usage
  Train distribution: mean=0.006473, median=0.001049, std=0.012184, max=0.218994

  Training avg_cpu_usage_q10 (quantile=0.1)... Done in 0.4s | Pinball=0.000545 | Coverage=10.0% (expected ~10%) | MAE=0.005439

  Training avg_cpu_usage_q50 (quantile=0.5)... Done in 0.7s | Pinball=0.002205 | Coverage=48.8% (expected ~50%) | MAE=0.004410

  Training avg_cpu_usage_q90 (quantile=0.9)... Done in 0.9s | Pinball=0.001348 | Coverage=86.3% (expected ~90%) | MAE=0.008552
Target: avg_memory_usage
  Train distribution: mean=0.004133, median=0.001099, std=0.007674, max=0.168701

  Training avg_memory_usage_q10 (quantile=0.1)... Done in 0.3s | Pinball=0.000299 | Coverage=7.9% (expected ~10%) | MAE=0.002975

  Training avg_memory_usage_q50 (quantile=0.5)... Done in 0.5s | Pinball=0.001249 | Coverage=51.4% (expected ~50%) | MAE=0.002498

  Training avg_memory_usage_q90 (quantile=0.9)... Done in 1.0s | Pinball=0.000706 | Coverage=90.8% (expected ~90%) | MAE

## Results Summary

In [34]:
print("TRAINING RESULTS SUMMARY")

results_df = pd.DataFrame(all_results)

# Pivot table for clean display
print("\n--- Pinball Loss (lower = better) ---")
pinball_pivot = results_df.pivot(index='target', columns='quantile', values='pinball_loss')
pinball_pivot.columns = [f"P{int(c*100)}" for c in pinball_pivot.columns]
print(pinball_pivot.to_string())

print("\n--- Coverage (should match expected quantile) ---")
coverage_pivot = results_df.pivot(index='target', columns='quantile', values='coverage')
coverage_pivot.columns = [f"P{int(c*100)}" for c in coverage_pivot.columns]
print(coverage_pivot.to_string())

print("\n--- Coverage Error (|actual - expected|, lower = better calibrated) ---")
coverage_err_pivot = results_df.pivot(index='target', columns='quantile', values='coverage_error')
coverage_err_pivot.columns = [f"P{int(c*100)}" for c in coverage_err_pivot.columns]
print(coverage_err_pivot.to_string())

# Save full results
results_path = os.path.join(RESULTS_DIR, "quantile_training_results.csv")
results_df.to_csv(results_path, index=False)
print(f"\nDetailed results saved to: {results_path}")

TRAINING RESULTS SUMMARY

--- Pinball Loss (lower = better) ---
                       P10       P50       P90
target                                        
avg_cpu_usage     0.000545  0.002205  0.001348
avg_memory_usage  0.000299  0.001249  0.000706
duration_seconds  0.021363  0.040315  0.008063
max_cpu_usage     0.002061  0.008330  0.005621

--- Coverage (should match expected quantile) ---
                       P10       P50       P90
target                                        
avg_cpu_usage     0.100054  0.487635  0.862582
avg_memory_usage  0.078664  0.514189  0.908490
duration_seconds  0.109078  1.000000  1.000000
max_cpu_usage     0.099823  0.429745  0.906474

--- Coverage Error (|actual - expected|, lower = better calibrated) ---
                       P10       P50       P90
target                                        
avg_cpu_usage     0.000054  0.012365  0.037418
avg_memory_usage  0.021336  0.014189  0.008490
duration_seconds  0.009078  0.500000  0.100000
max_cpu_usage

## FEATURE IMPORTANCE 

In [35]:
print("FEATURE IMPORTANCE (from P50 / Median models)")
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, target in enumerate(QUANTILE_TARGETS):
    model_key = f"{target}_q50"
    model = trained_models[model_key]
    
    importance = model.feature_importances_
    feature_names = X_train.columns
    
    sorted_idx = np.argsort(importance)
    
    axes[idx].barh(range(len(sorted_idx)), importance[sorted_idx], color='steelblue')
    axes[idx].set_yticks(range(len(sorted_idx)))
    axes[idx].set_yticklabels(feature_names[sorted_idx])
    axes[idx].set_title(f'{target} (P50)', fontweight='bold')
    axes[idx].set_xlabel('Feature Importance (Split Count)')

plt.suptitle('Feature Importance for Quantile Regression Models', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "feature_importance.png"), dpi=150)
plt.close()
print("Feature importance plot saved to training_results/feature_importance.png")


FEATURE IMPORTANCE (from P50 / Median models)
Feature importance plot saved to training_results/feature_importance.png


## QUANTILE CROSSING CHECK

In [36]:
#  QUANTILE CROSSING CHECK
print("\n\n" + "=" * 70)
print("QUANTILE CROSSING VALIDATION")
print("=" * 70)
print("Checking P10 <= P50 <= P90 for all predictions on the test set...\n")

crossing_issues = {}
for target in QUANTILE_TARGETS:
    p10_preds = trained_models[f"{target}_q10"].predict(X_test)
    p50_preds = trained_models[f"{target}_q50"].predict(X_test)
    p90_preds = trained_models[f"{target}_q90"].predict(X_test)
    
    # Check for violations
    crossing_10_50 = np.sum(p10_preds > p50_preds)
    crossing_50_90 = np.sum(p50_preds > p90_preds)
    total = len(p10_preds)
    
    crossing_issues[target] = {
        'p10_gt_p50': crossing_10_50,
        'p50_gt_p90': crossing_50_90,
        'total_samples': total,
        'p10_gt_p50_pct': crossing_10_50 / total * 100,
        'p50_gt_p90_pct': crossing_50_90 / total * 100
    }
    
    status_10_50 = "✓ OK" if crossing_10_50 == 0 else f"⚠ {crossing_10_50:,} violations ({crossing_10_50/total*100:.2f}%)"
    status_50_90 = "✓ OK" if crossing_50_90 == 0 else f"⚠ {crossing_50_90:,} violations ({crossing_50_90/total*100:.2f}%)"
    
    print(f"{target}:")
    print(f"  P10 > P50: {status_10_50}")
    print(f"  P50 > P90: {status_50_90}")




QUANTILE CROSSING VALIDATION
Checking P10 <= P50 <= P90 for all predictions on the test set...

avg_cpu_usage:
  P10 > P50: ⚠ 1 violations (0.00%)
  P50 > P90: ⚠ 6 violations (0.01%)
avg_memory_usage:
  P10 > P50: ✓ OK
  P50 > P90: ⚠ 3,429 violations (6.58%)
max_cpu_usage:
  P10 > P50: ⚠ 1 violations (0.00%)
  P50 > P90: ⚠ 1 violations (0.00%)
duration_seconds:
  P10 > P50: ✓ OK
  P50 > P90: ✓ OK


In [37]:

print("SAVING MODEL METADATA")

model_metadata = {
    'targets': QUANTILE_TARGETS,
    'quantiles': QUANTILES,
    'input_features': list(X_train.columns),
    'n_features': X_train.shape[1],
    'n_train_samples': X_train.shape[0],
    'n_test_samples': X_test.shape[0],
    'model_files': {},
    'training_metrics': {}
}

for r in all_results:
    model_metadata['model_files'][r['model_key']] = f"{r['model_key']}.pkl"
    model_metadata['training_metrics'][r['model_key']] = {
        'pinball_loss': round(r['pinball_loss'], 8),
        'coverage': round(r['coverage'], 4),
        'mae': round(r['mae'], 8),
        'best_iteration': r['best_iteration']
    }

metadata_path = os.path.join(MODEL_DIR, "model_metadata.json")
with open(metadata_path, 'w') as f:
    json.dump(model_metadata, f, indent=4)

print(f"Model metadata saved to: {metadata_path}")

# List all saved model files
print(f"\nSaved {len(all_results)} model files to '{MODEL_DIR}/':")
for r in all_results:
    print(f"  ├── {r['model_key']}.pkl")
print(f"  └── model_metadata.json")




SAVING MODEL METADATA
Model metadata saved to: trained_models/model_metadata.json

Saved 12 model files to 'trained_models/':
  ├── avg_cpu_usage_q10.pkl
  ├── avg_cpu_usage_q50.pkl
  ├── avg_cpu_usage_q90.pkl
  ├── avg_memory_usage_q10.pkl
  ├── avg_memory_usage_q50.pkl
  ├── avg_memory_usage_q90.pkl
  ├── max_cpu_usage_q10.pkl
  ├── max_cpu_usage_q50.pkl
  ├── max_cpu_usage_q90.pkl
  ├── duration_seconds_q10.pkl
  ├── duration_seconds_q50.pkl
  ├── duration_seconds_q90.pkl
  └── model_metadata.json


# XGBoost Quantile Regression Model

In [38]:
import xgboost as xgb
FEATURES_DIR = "/kaggle/working/features"
MODEL_DIR = "trained_models_xgb"       # Separate directory from LightGBM
RESULTS_DIR = "training_results_xgb"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

In [39]:
# Same targets and quantiles as LightGBM for fair comparison
QUANTILE_TARGETS = [
    'avg_cpu_usage',
    'avg_memory_usage',
    'max_cpu_usage',
    'duration_seconds'
]
QUANTILES = [0.10, 0.50, 0.90]

In [40]:
# XGBoost base hyperparameters (matched to LightGBM complexity for fair comparison)
BASE_PARAMS = {
    'n_estimators': 500,
    'learning_rate': 0.05,
    'max_depth': 7,
    'min_child_weight': 50,       # analogous to LightGBM's min_child_samples
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,             # L1
    'reg_lambda': 1.0,            # L2
    'random_state': 42,
    'verbosity': 0,
    'n_jobs': -1,
    'tree_method': 'hist',        # fast histogram-based method (like LightGBM)
}

print(f"\nTargets: {QUANTILE_TARGETS}")
print(f"Quantiles: {QUANTILES}")
print(f"Total models to train: {len(QUANTILE_TARGETS) * len(QUANTILES)}")



Targets: ['avg_cpu_usage', 'avg_memory_usage', 'max_cpu_usage', 'duration_seconds']
Quantiles: [0.1, 0.5, 0.9]
Total models to train: 12


In [41]:
#LOAD DATA
print("\n--- Loading Data ---")
X_train = pd.read_csv(os.path.join(FEATURES_DIR, "X_train.csv"))
X_test = pd.read_csv(os.path.join(FEATURES_DIR, "X_test.csv"))
y_train = pd.read_csv(os.path.join(FEATURES_DIR, "y_train.csv"))
y_test = pd.read_csv(os.path.join(FEATURES_DIR, "y_test.csv"))

print(f"Training set: {X_train.shape[0]:,} rows × {X_train.shape[1]} features")
print(f"Testing set:  {X_test.shape[0]:,} rows × {X_test.shape[1]} features")

assert X_train.shape[0] == y_train.shape[0], "X/y train mismatch!"
assert X_test.shape[0] == y_test.shape[0], "X/y test mismatch!"
print("Data integrity checks passed.")


--- Loading Data ---
Training set: 208,324 rows × 13 features
Testing set:  52,082 rows × 13 features
Data integrity checks passed.


In [42]:
# TRAIN ALL QUANTILE MODELS
print("TRAINING XGBOOST QUANTILE MODELS")

all_results = []
trained_models = {}
total_start = time.time()

for target_name in QUANTILE_TARGETS:
    print(f"\n{'─' * 50}")
    print(f"Target: {target_name}")
    print(f"{'─' * 50}")
    
    y_tr = y_train[target_name].values
    y_te = y_test[target_name].values
    
    print(f"  Train distribution: mean={y_tr.mean():.6f}, median={np.median(y_tr):.6f}, "
          f"std={y_tr.std():.6f}, max={y_tr.max():.6f}")
    
    for q in QUANTILES:
        model_key = f"xgb_{target_name}_q{int(q*100)}"
        print(f"\n  Training {model_key} (quantile={q})...", end=" ")
        
        start_time = time.time()
        
        # XGBoost quantile regression objective
        params = BASE_PARAMS.copy()
        params['objective'] = 'reg:quantileerror'
        params['quantile_alpha'] = q   # The quantile level (tau)
        
        model = xgb.XGBRegressor(**params)
        
        # Train with early stopping
        model.fit(
            X_train, y_tr,
            eval_set=[(X_test, y_te)],
            verbose=False
        )
        
        train_time = time.time() - start_time
        
        # Predictions
        y_pred = model.predict(X_test)
        
        # --- Evaluation Metrics (same as LightGBM for fair comparison) ---
        errors = y_te - y_pred
        pinball_loss = np.mean(np.where(errors >= 0, q * errors, (q - 1) * errors))
        coverage = np.mean(y_te <= y_pred)
        mae = np.mean(np.abs(errors))
        
        print(f"Done in {train_time:.1f}s | "
              f"Pinball={pinball_loss:.6f} | "
              f"Coverage={coverage:.1%} (expected ~{q:.0%}) | "
              f"MAE={mae:.6f}")
        
        # Save model as .pkl
        model_path = os.path.join(MODEL_DIR, f"{model_key}.pkl")
        joblib.dump(model, model_path)
        
        all_results.append({
            'target': target_name,
            'quantile': q,
            'model_key': model_key,
            'pinball_loss': pinball_loss,
            'coverage': coverage,
            'expected_coverage': q,
            'coverage_error': abs(coverage - q),
            'mae': mae,
            'train_time_seconds': train_time,
            'best_iteration': model.best_iteration if hasattr(model, 'best_iteration') else params['n_estimators'],
            'n_features': X_train.shape[1]
        })
        
        trained_models[model_key] = model

total_time = time.time() - total_start
print(f"Total training time: {total_time:.1f} seconds")


TRAINING XGBOOST QUANTILE MODELS

──────────────────────────────────────────────────
Target: avg_cpu_usage
──────────────────────────────────────────────────
  Train distribution: mean=0.006473, median=0.001049, std=0.012184, max=0.218994

  Training xgb_avg_cpu_usage_q10 (quantile=0.1)... Done in 10.9s | Pinball=0.000695 | Coverage=16.3% (expected ~10%) | MAE=0.005086

  Training xgb_avg_cpu_usage_q50 (quantile=0.5)... Done in 10.9s | Pinball=0.002439 | Coverage=42.3% (expected ~50%) | MAE=0.004877

  Training xgb_avg_cpu_usage_q90 (quantile=0.9)... Done in 10.9s | Pinball=0.001453 | Coverage=81.2% (expected ~90%) | MAE=0.008094

──────────────────────────────────────────────────
Target: avg_memory_usage
──────────────────────────────────────────────────
  Train distribution: mean=0.004133, median=0.001099, std=0.007674, max=0.168701

  Training xgb_avg_memory_usage_q10 (quantile=0.1)... Done in 10.7s | Pinball=0.000523 | Coverage=21.2% (expected ~10%) | MAE=0.002625

  Training xgb_a

## RESULTS SUMMARY

In [43]:
print("XGBOOST TRAINING RESULTS SUMMARY")

results_df = pd.DataFrame(all_results)

print("\n--- Pinball Loss (lower = better) ---")
pinball_pivot = results_df.pivot(index='target', columns='quantile', values='pinball_loss')
pinball_pivot.columns = [f"P{int(c*100)}" for c in pinball_pivot.columns]
print(pinball_pivot.to_string())

print("\n--- Coverage (should match expected quantile) ---")
coverage_pivot = results_df.pivot(index='target', columns='quantile', values='coverage')
coverage_pivot.columns = [f"P{int(c*100)}" for c in coverage_pivot.columns]
print(coverage_pivot.to_string())

print("\n--- Coverage Error (|actual - expected|, lower = better calibrated) ---")
coverage_err_pivot = results_df.pivot(index='target', columns='quantile', values='coverage_error')
coverage_err_pivot.columns = [f"P{int(c*100)}" for c in coverage_err_pivot.columns]
print(coverage_err_pivot.to_string())

# Save full results
results_path = os.path.join(RESULTS_DIR, "xgb_quantile_training_results.csv")
results_df.to_csv(results_path, index=False)
print(f"\nResults saved to: {results_path}")

XGBOOST TRAINING RESULTS SUMMARY

--- Pinball Loss (lower = better) ---
                       P10       P50       P90
target                                        
avg_cpu_usage     0.000695  0.002439  0.001453
avg_memory_usage  0.000523  0.001258  0.000787
duration_seconds  0.031128  0.040315  0.008063
max_cpu_usage     0.002438  0.008519  0.005868

--- Coverage (should match expected quantile) ---
                       P10       P50       P90
target                                        
avg_cpu_usage     0.162974  0.422680  0.812181
avg_memory_usage  0.211762  0.357974  0.795918
duration_seconds  0.204351  1.000000  1.000000
max_cpu_usage     0.174436  0.407799  0.821397

--- Coverage Error (|actual - expected|, lower = better calibrated) ---
                       P10       P50       P90
target                                        
avg_cpu_usage     0.062974  0.077320  0.087819
avg_memory_usage  0.111762  0.142026  0.104082
duration_seconds  0.104351  0.500000  0.100000
max_c

## FEATURE IMPORTANCE

In [44]:
print("FEATURE IMPORTANCE (XGBoost P50 models)")

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, target in enumerate(QUANTILE_TARGETS):
    model_key = f"xgb_{target}_q50"
    model = trained_models[model_key]
    
    importance = model.feature_importances_
    feature_names = X_train.columns
    sorted_idx = np.argsort(importance)
    
    axes[idx].barh(range(len(sorted_idx)), importance[sorted_idx], color='#e74c3c')
    axes[idx].set_yticks(range(len(sorted_idx)))
    axes[idx].set_yticklabels(feature_names[sorted_idx])
    axes[idx].set_title(f'{target} (P50) - XGBoost', fontweight='bold')
    axes[idx].set_xlabel('Feature Importance')

plt.suptitle('XGBoost Feature Importance for Quantile Regression', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "xgb_feature_importance.png"), dpi=150)
plt.close()
print("Feature importance plot saved.")

FEATURE IMPORTANCE (XGBoost P50 models)
Feature importance plot saved.


## QUANTILE CROSSING CHECK

In [45]:
print("QUANTILE CROSSING VALIDATION (XGBoost)")

print("Checking P10 <= P50 <= P90...\n")

for target in QUANTILE_TARGETS:
    p10 = trained_models[f"xgb_{target}_q10"].predict(X_test)
    p50 = trained_models[f"xgb_{target}_q50"].predict(X_test)
    p90 = trained_models[f"xgb_{target}_q90"].predict(X_test)
    c_10_50 = np.sum(p10 > p50)
    c_50_90 = np.sum(p50 > p90)
    total = len(p10)
    
    s1 = "✓ OK" if c_10_50 == 0 else f"⚠ {c_10_50:,} violations ({c_10_50/total*100:.2f}%)"
    s2 = "✓ OK" if c_50_90 == 0 else f"⚠ {c_50_90:,} violations ({c_50_90/total*100:.2f}%)"
    
    print(f"{target}:")
    print(f"  P10 > P50: {s1}")
    print(f"  P50 > P90: {s2}")


QUANTILE CROSSING VALIDATION (XGBoost)
Checking P10 <= P50 <= P90...

avg_cpu_usage:
  P10 > P50: ⚠ 593 violations (1.14%)
  P50 > P90: ⚠ 2,494 violations (4.79%)
avg_memory_usage:
  P10 > P50: ⚠ 3,976 violations (7.63%)
  P50 > P90: ⚠ 1,530 violations (2.94%)
max_cpu_usage:
  P10 > P50: ⚠ 1,110 violations (2.13%)
  P50 > P90: ⚠ 2,816 violations (5.41%)
duration_seconds:
  P10 > P50: ⚠ 1,046 violations (2.01%)
  P50 > P90: ✓ OK


## SAVE MODEL METADATA

In [46]:
print("\n\n" + "=" * 70)
print("SAVING XGBoost MODEL METADATA")
print("=" * 70)

model_metadata = {
    'framework': 'xgboost',
    'targets': QUANTILE_TARGETS,
    'quantiles': QUANTILES,
    'input_features': list(X_train.columns),
    'n_features': X_train.shape[1],
    'n_train_samples': X_train.shape[0],
    'n_test_samples': X_test.shape[0],
    'model_files': {},
    'training_metrics': {}
}

for r in all_results:
    model_metadata['model_files'][r['model_key']] = f"{r['model_key']}.pkl"
    model_metadata['training_metrics'][r['model_key']] = {
        'pinball_loss': round(r['pinball_loss'], 8),
        'coverage': round(r['coverage'], 4),
        'mae': round(r['mae'], 8),
        'best_iteration': r['best_iteration']
    }

metadata_path = os.path.join(MODEL_DIR, "model_metadata.json")
with open(metadata_path, 'w') as f:
    json.dump(model_metadata, f, indent=4)

print(f"Model metadata saved to: {metadata_path}")
print(f"\nSaved {len(all_results)} model files to '{MODEL_DIR}/':")
for r in all_results:
    print(f"  ├── {r['model_key']}.pkl")
print(f"  |── model_metadata.json")




SAVING XGBoost MODEL METADATA
Model metadata saved to: trained_models_xgb/model_metadata.json

Saved 12 model files to 'trained_models_xgb/':
  ├── xgb_avg_cpu_usage_q10.pkl
  ├── xgb_avg_cpu_usage_q50.pkl
  ├── xgb_avg_cpu_usage_q90.pkl
  ├── xgb_avg_memory_usage_q10.pkl
  ├── xgb_avg_memory_usage_q50.pkl
  ├── xgb_avg_memory_usage_q90.pkl
  ├── xgb_max_cpu_usage_q10.pkl
  ├── xgb_max_cpu_usage_q50.pkl
  ├── xgb_max_cpu_usage_q90.pkl
  ├── xgb_duration_seconds_q10.pkl
  ├── xgb_duration_seconds_q50.pkl
  ├── xgb_duration_seconds_q90.pkl
  |── model_metadata.json


# Calibration Validation — LightGBM vs XGBoost Comparison

In [47]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
FEATURES_DIR = "/kaggle/working/features"
LGBM_MODEL_DIR = "/kaggle/working/trained_models"
XGB_MODEL_DIR = "/kaggle/working/trained_models_xgb"
OUTPUT_DIR = "validation_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [48]:
QUANTILE_TARGETS = ['avg_cpu_usage', 'avg_memory_usage', 'max_cpu_usage', 'duration_seconds']
QUANTILES = [0.10, 0.50, 0.90]
QUANTILE_LABELS = {0.10: 'P10', 0.50: 'P50', 0.90: 'P90'}

In [49]:
#LOAD DATA
print("\n--- Loading Test Data ---")
X_test = pd.read_csv(os.path.join(FEATURES_DIR, "X_test.csv"))
y_test = pd.read_csv(os.path.join(FEATURES_DIR, "y_test.csv"))
print(f"Test set: {X_test.shape[0]:,} rows × {X_test.shape[1]} features")



--- Loading Test Data ---
Test set: 52,082 rows × 13 features


In [50]:
#  LOAD ALL MODELS
print("\n--- Loading Trained Models ---")

lgbm_models = {}
xgb_models = {}

for target in QUANTILE_TARGETS:
    for q in QUANTILES:
        key = f"{target}_q{int(q*100)}"
        xgb_key = f"xgb_{target}_q{int(q*100)}"
        
        # Load LightGBM
        lgbm_path = os.path.join(LGBM_MODEL_DIR, f"{key}.pkl")
        if os.path.exists(lgbm_path):
            lgbm_models[key] = joblib.load(lgbm_path)
        else:
            print(f"  ⚠ LightGBM model not found: {lgbm_path}")
        
        # Load XGBoost (files are prefixed with xgb_)
        xgb_path = os.path.join(XGB_MODEL_DIR, f"{xgb_key}.pkl")
        if os.path.exists(xgb_path):
            xgb_models[key] = joblib.load(xgb_path)  # store with same key for easy comparison
        else:
            print(f"  ⚠ XGBoost model not found: {xgb_path}")

print(f"  Loaded {len(lgbm_models)} LightGBM models")
print(f"  Loaded {len(xgb_models)} XGBoost models")


--- Loading Trained Models ---
  Loaded 12 LightGBM models
  Loaded 12 XGBoost models


In [51]:
# COMPUTE CALIBRATION METRICS

print("CALIBRATION METRICS COMPARISON")

comparison_rows = []

for target in QUANTILE_TARGETS:
    y_true = y_test[target].values
    
    for q in QUANTILES:
        key = f"{target}_q{int(q*100)}"
        row = {'target': target, 'quantile': q, 'quantile_label': QUANTILE_LABELS[q]}
        print(f"  Evaluating {target} ({QUANTILE_LABELS[q]})...", end=" ")
        # --- LightGBM ---
        if key in lgbm_models:
            lgbm_pred = lgbm_models[key].predict(X_test)
            lgbm_errors = y_true - lgbm_pred
            
            row['lgbm_coverage'] = np.mean(y_true <= lgbm_pred)
            row['lgbm_pinball'] = np.mean(np.where(lgbm_errors >= 0, q * lgbm_errors, (q - 1) * lgbm_errors))
            row['lgbm_mae'] = np.mean(np.abs(lgbm_errors))
            row['lgbm_rmse'] = np.sqrt(np.mean(lgbm_errors ** 2))
            row['lgbm_coverage_error'] = abs(row['lgbm_coverage'] - q)
        
        # --- XGBoost ---
        if key in xgb_models:
            xgb_pred = xgb_models[key].predict(X_test)
            xgb_errors = y_true - xgb_pred
            
            row['xgb_coverage'] = np.mean(y_true <= xgb_pred)
            row['xgb_pinball'] = np.mean(np.where(xgb_errors >= 0, q * xgb_errors, (q - 1) * xgb_errors))
            row['xgb_mae'] = np.mean(np.abs(xgb_errors))
            row['xgb_rmse'] = np.sqrt(np.mean(xgb_errors ** 2))
            row['xgb_coverage_error'] = abs(row['xgb_coverage'] - q)
        print("Done.")
        comparison_rows.append(row)

comp_df = pd.DataFrame(comparison_rows)


CALIBRATION METRICS COMPARISON
  Evaluating avg_cpu_usage (P10)... Done.
  Evaluating avg_cpu_usage (P50)... Done.
  Evaluating avg_cpu_usage (P90)... Done.
  Evaluating avg_memory_usage (P10)... Done.
  Evaluating avg_memory_usage (P50)... Done.
  Evaluating avg_memory_usage (P90)... Done.
  Evaluating max_cpu_usage (P10)... Done.
  Evaluating max_cpu_usage (P50)... Done.
  Evaluating max_cpu_usage (P90)... Done.
  Evaluating duration_seconds (P10)... Done.
  Evaluating duration_seconds (P50)... Done.
  Evaluating duration_seconds (P90)... Done.


## COMPARISON TABLES

In [52]:

# --- Coverage Comparison ---

print("COVERAGE COMPARISON (closer to expected quantile = better)")

print(f"{'Target':<22} {'Q':>4} {'Expected':>9} {'LightGBM':>10} {'XGBoost':>10} {'Winner':>10}")

for _, r in comp_df.iterrows():
    lgbm_cov = r.get('lgbm_coverage', float('nan'))
    xgb_cov = r.get('xgb_coverage', float('nan'))
    lgbm_err = abs(lgbm_cov - r['quantile'])
    xgb_err = abs(xgb_cov - r['quantile'])
    winner = "LightGBM" if lgbm_err < xgb_err else "XGBoost" if xgb_err < lgbm_err else "Tie"
    print(f"{r['target']:<22} {r['quantile_label']:>4} {r['quantile']:>8.0%} "
          f"{lgbm_cov:>9.2%} {xgb_cov:>9.2%} {winner:>10}")

# --- Pinball Loss Comparison ---

print("PINBALL LOSS COMPARISON (lower = better)")
print(f"{'Target':<22} {'Q':>4} {'LightGBM':>12} {'XGBoost':>12} {'Winner':>10}")
for _, r in comp_df.iterrows():
    lgbm_pb = r.get('lgbm_pinball', float('nan'))
    xgb_pb = r.get('xgb_pinball', float('nan'))
    winner = "LightGBM" if lgbm_pb < xgb_pb else "XGBoost" if xgb_pb < lgbm_pb else "Tie"
    print(f"{r['target']:<22} {r['quantile_label']:>4} {lgbm_pb:>12.8f} {xgb_pb:>12.8f} {winner:>10}")

# --- MAE Comparison ---

print("MAE COMPARISON (lower = better)")

print(f"{'Target':<22} {'Q':>4} {'LightGBM':>12} {'XGBoost':>12} {'Winner':>10}")
for _, r in comp_df.iterrows():
    lgbm_mae = r.get('lgbm_mae', float('nan'))
    xgb_mae = r.get('xgb_mae', float('nan'))
    winner = "LightGBM" if lgbm_mae < xgb_mae else "XGBoost" if xgb_mae < lgbm_mae else "Tie"
    print(f"{r['target']:<22} {r['quantile_label']:>4} {lgbm_mae:>12.8f} {xgb_mae:>12.8f} {winner:>10}")

# --- Overall Scorecard ---
print("OVERALL SCORECARD")


lgbm_wins = 0
xgb_wins = 0
ties = 0

for _, r in comp_df.iterrows():
    # Score based on pinball loss (the proper quantile metric)
    lgbm_pb = r.get('lgbm_pinball', float('inf'))
    xgb_pb = r.get('xgb_pinball', float('inf'))
    if lgbm_pb < xgb_pb:
        lgbm_wins += 1
    elif xgb_pb < lgbm_pb:
        xgb_wins += 1
    else:
        ties += 1

total = lgbm_wins + xgb_wins + ties
print(f"Based on Pinball Loss :")
print(f"  LightGBM wins: {lgbm_wins}/{total} ({lgbm_wins/total*100:.0f}%)")
print(f"  XGBoost wins:  {xgb_wins}/{total} ({xgb_wins/total*100:.0f}%)")
print(f"  Ties:          {ties}/{total}")
# Save comparison CSV
comp_df.to_csv(os.path.join(OUTPUT_DIR, "lgbm_vs_xgb_comparison.csv"), index=False)
print(f"\nFull comparison saved to: {OUTPUT_DIR}/lgbm_vs_xgb_comparison.csv")

COVERAGE COMPARISON (closer to expected quantile = better)
Target                    Q  Expected   LightGBM    XGBoost     Winner
avg_cpu_usage           P10      10%    10.01%    16.30%   LightGBM
avg_cpu_usage           P50      50%    48.76%    42.27%   LightGBM
avg_cpu_usage           P90      90%    86.26%    81.22%   LightGBM
avg_memory_usage        P10      10%     7.87%    21.18%   LightGBM
avg_memory_usage        P50      50%    51.42%    35.80%   LightGBM
avg_memory_usage        P90      90%    90.85%    79.59%   LightGBM
max_cpu_usage           P10      10%     9.98%    17.44%   LightGBM
max_cpu_usage           P50      50%    42.97%    40.78%   LightGBM
max_cpu_usage           P90      90%    90.65%    82.14%   LightGBM
duration_seconds        P10      10%    10.91%    20.44%   LightGBM
duration_seconds        P50      50%   100.00%   100.00%        Tie
duration_seconds        P90      90%   100.00%   100.00%        Tie
PINBALL LOSS COMPARISON (lower = better)
Target       

## VISUALIZATION: CALIBRATION PLOTS

In [53]:

# --- Plot 1: Coverage Calibration Chart ---
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for idx, target in enumerate(QUANTILE_TARGETS):
    ax = axes[idx]
    target_data = comp_df[comp_df['target'] == target]
    
    qs = [r['quantile'] for _, r in target_data.iterrows()]
    lgbm_covs = [r.get('lgbm_coverage', np.nan) for _, r in target_data.iterrows()]
    xgb_covs = [r.get('xgb_coverage', np.nan) for _, r in target_data.iterrows()]
    
    x = np.arange(len(qs))
    width = 0.3
    
    bars1 = ax.bar(x - width/2, lgbm_covs, width, label='LightGBM', color='#3498db', alpha=0.85)
    bars2 = ax.bar(x + width/2, xgb_covs, width, label='XGBoost', color='#e74c3c', alpha=0.85)
    
    # Draw expected coverage lines
    for i, q in enumerate(qs):
        ax.hlines(q, i - 0.4, i + 0.4, colors='green', linestyles='dashed', linewidth=1.5)
    
    ax.set_xticks(x)
    ax.set_xticklabels([QUANTILE_LABELS[q] for q in qs])
    ax.set_title(target, fontweight='bold', fontsize=11)
    ax.set_ylabel('Actual Coverage')
    ax.set_ylim(0, 1.15)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Coverage Calibration: LightGBM vs XGBoost\n(Green dashed = expected coverage)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "coverage_calibration_comparison.png"), dpi=150)
plt.show()
plt.close()
print(" Coverage calibration plot saved.")

# --- Plot 2: Pinball Loss Comparison ---
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for idx, target in enumerate(QUANTILE_TARGETS):
    ax = axes[idx]
    target_data = comp_df[comp_df['target'] == target]
    
    qs = [r['quantile'] for _, r in target_data.iterrows()]
    lgbm_pbs = [r.get('lgbm_pinball', np.nan) for _, r in target_data.iterrows()]
    xgb_pbs = [r.get('xgb_pinball', np.nan) for _, r in target_data.iterrows()]
    
    x = np.arange(len(qs))
    width = 0.3
    
    ax.bar(x - width/2, lgbm_pbs, width, label='LightGBM', color='#3498db', alpha=0.85)
    ax.bar(x + width/2, xgb_pbs, width, label='XGBoost', color='#e74c3c', alpha=0.85)
    
    ax.set_xticks(x)
    ax.set_xticklabels([QUANTILE_LABELS[q] for q in qs])
    ax.set_title(target, fontweight='bold', fontsize=11)
    ax.set_ylabel('Pinball Loss')
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Pinball Loss: LightGBM vs XGBoost\n(Lower = Better Quantile Estimation)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "pinball_loss_comparison.png"), dpi=150)
plt.show()
plt.close()
print(" Pinball loss comparison plot saved.")

# --- Plot 3: Prediction Interval Width (P90 - P10) ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, target in enumerate(QUANTILE_TARGETS):
    ax = axes[idx]
    y_true = y_test[target].values
    
    # LightGBM interval
    lgbm_p10 = lgbm_models[f"{target}_q10"].predict(X_test)
    lgbm_p90 = lgbm_models[f"{target}_q90"].predict(X_test)
    lgbm_width = lgbm_p90 - lgbm_p10
    
    # XGBoost interval
    xgb_p10 = xgb_models[f"{target}_q10"].predict(X_test)
    xgb_p90 = xgb_models[f"{target}_q90"].predict(X_test)
    xgb_width = xgb_p90 - xgb_p10
    
    ax.hist(lgbm_width, bins=50, alpha=0.6, label=f'LightGBM (mean={lgbm_width.mean():.4f})', color='#3498db')
    ax.hist(xgb_width, bins=50, alpha=0.6, label=f'XGBoost (mean={xgb_width.mean():.4f})', color='#e74c3c')
    ax.set_title(f'{target}\nPrediction Interval Width (P90 - P10)', fontweight='bold')
    ax.set_xlabel('Interval Width')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle('Prediction Interval Width Distribution\n(Narrower = More Confident, Wider = More Conservative)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "interval_width_comparison.png"), dpi=150)
plt.show()
plt.close()
print("Interval width comparison plot saved.")

# --- Plot 4: Actual vs Predicted (P50) scatter for each target ---
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for idx, target in enumerate(QUANTILE_TARGETS):
    y_true = y_test[target].values
    
    # Sample 2000 points for cleaner scatter plot
    sample_idx = np.random.RandomState(42).choice(len(y_true), size=min(2000, len(y_true)), replace=False)
    y_sample = y_true[sample_idx]
    
    # LightGBM P50
    ax = axes[0, idx]
    lgbm_p50 = lgbm_models[f"{target}_q50"].predict(X_test)[sample_idx]
    ax.scatter(y_sample, lgbm_p50, alpha=0.3, s=10, color='#3498db')
    lims = [min(y_sample.min(), lgbm_p50.min()), max(y_sample.max(), lgbm_p50.max())]
    ax.plot(lims, lims, 'k--', alpha=0.5, label='Perfect')
    ax.set_title(f'LightGBM - {target}', fontweight='bold', fontsize=10)
    ax.set_xlabel('Actual')
    ax.set_ylabel('Predicted (P50)')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    
    # XGBoost P50
    ax = axes[1, idx]
    xgb_p50 = xgb_models[f"{target}_q50"].predict(X_test)[sample_idx]
    ax.scatter(y_sample, xgb_p50, alpha=0.3, s=10, color='#e74c3c')
    lims = [min(y_sample.min(), xgb_p50.min()), max(y_sample.max(), xgb_p50.max())]
    ax.plot(lims, lims, 'k--', alpha=0.5, label='Perfect')
    ax.set_title(f'XGBoost - {target}', fontweight='bold', fontsize=10)
    ax.set_xlabel('Actual')
    ax.set_ylabel('Predicted (P50)')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Actual vs Predicted (P50 / Median)\nPoints on dashed line = perfect prediction', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "actual_vs_predicted_comparison.png"), dpi=150)
plt.show()
plt.close()
print("Actual vs predicted scatter plot saved.")


 Coverage calibration plot saved.
 Pinball loss comparison plot saved.
Interval width comparison plot saved.
Actual vs predicted scatter plot saved.


## FINAL RECOMMENDATION


In [54]:
print("FINAL SUMMARY & RECOMMENDATION")

# Calculate aggregate metrics
lgbm_avg_coverage_err = comp_df['lgbm_coverage_error'].mean()
xgb_avg_coverage_err = comp_df['xgb_coverage_error'].mean()
lgbm_avg_pinball = comp_df['lgbm_pinball'].mean()
xgb_avg_pinball = comp_df['xgb_pinball'].mean()

print(f"\n  Metric                    LightGBM     XGBoost")
print(f"  ─────────────────────────────────────────────")
print(f"  Avg Coverage Error:       {lgbm_avg_coverage_err:.4f}       {xgb_avg_coverage_err:.4f}")
print(f"  Avg Pinball Loss:         {lgbm_avg_pinball:.6f}     {xgb_avg_pinball:.6f}")
print(f"  Wins (by Pinball):        {lgbm_wins}/12         {xgb_wins}/12")

if lgbm_avg_pinball < xgb_avg_pinball:
    print(f"\n  → Recommendation: LightGBM (lower average pinball loss)")
elif xgb_avg_pinball < lgbm_avg_pinball:
    print(f"\n  → Recommendation: XGBoost (lower average pinball loss)")
else:
    print(f"\n  → Recommendation: Both models perform equally well")

print(f"\nAll comparison results and plots saved to '{OUTPUT_DIR}/':")
print(f"  ├── lgbm_vs_xgb_comparison.csv")
print(f"  ├── coverage_calibration_comparison.png")
print(f"  ├── pinball_loss_comparison.png")
print(f"  ├── interval_width_comparison.png")
print(f"  └── actual_vs_predicted_comparison.png")


FINAL SUMMARY & RECOMMENDATION

  Metric                    LightGBM     XGBoost
  ─────────────────────────────────────────────
  Avg Coverage Error:       0.0650       0.1280
  Avg Pinball Loss:         0.007675     0.008624
  Wins (by Pinball):        10/12         0/12

  → Recommendation: LightGBM (lower average pinball loss)

All comparison results and plots saved to 'validation_results/':
  ├── lgbm_vs_xgb_comparison.csv
  ├── coverage_calibration_comparison.png
  ├── pinball_loss_comparison.png
  ├── interval_width_comparison.png
  └── actual_vs_predicted_comparison.png
